# Bookshop Market Basket Analysis with Apache Spark FP-Growth

This notebook creates a simulated bookshop transaction dataset with millions of records, runs FP-Growth in Apache Spark, and extracts frequent itemsets and association rules.

The business focus is a bookshop selling items such as books, pens, notebooks, highlighters, bookmarks, sticky notes, planners, and erasers.

In [ ]:
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.ml.fpm import FPGrowth

spark = (
    SparkSession.builder
    .appName("BookshopFPGrowth")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

## 1. Simulated Bookshop Dataset

The dataset is generated at scale using Spark. Each transaction includes:
- `transaction_id`
- `timestamp`
- `store_id`
- `customer_type`
- `items` basket for market basket analysis

The generation logic intentionally creates realistic bookshop bundles such as:
- book + bookmark
- notebook + pen + highlighter
- planner + pen
- textbook + notebook + highlighter
- pencil + eraser
- diary + pen

In [ ]:
num_transactions = 1_000_000
seed = 42

templates = [
    ("books", ["book", "bookmark"]),
    ("stationery", ["notebook", "pen", "highlighter"]),
    ("study_set", ["textbook", "notebook", "highlighter"]),
    ("office", ["planner", "pen"]),
    ("school", ["pencil", "eraser", "notebook"]),
    ("gift", ["novel", "bookmark", "gift_wrap"]),
    ("reading", ["novel", "book"]),
    ("notes", ["sticky_notes", "pen", "notebook"]),
    ("journaling", ["diary", "pen"]),
    ("mixed", ["book", "notebook", "pen"]),
]

indexed_templates = spark.createDataFrame(
    [(i, t[0], t[1]) for i, t in enumerate(templates)],
    ["basket_pick", "basket_type", "template_items"]
)

base = spark.range(num_transactions).withColumnRenamed("id", "transaction_id")

transactions = (
    base
    .withColumn("store_id", (F.pmod(F.hash(F.col("transaction_id")), F.lit(12)) + 1).cast("int"))
    .withColumn("customer_rand", F.rand(seed + 1))
    .withColumn(
        "customer_type",
        F.when(F.col("customer_rand") < 0.55, F.lit("student"))
         .when(F.col("customer_rand") < 0.8, F.lit("office_worker"))
         .otherwise(F.lit("casual_reader"))
    )
    .withColumn(
        "timestamp",
        F.to_timestamp(
            F.from_unixtime(
                F.unix_timestamp(F.lit("2024-01-01 00:00:00"))
                + F.floor(F.rand(seed + 2) * F.lit(31536000)).cast("int")
            )
        )
    )
    .withColumn("basket_pick", F.floor(F.rand(seed + 3) * F.lit(len(templates))).cast("int"))
    .join(F.broadcast(indexed_templates), on="basket_pick", how="left")
    .withColumn(
        "basket_noise",
        F.array(
            F.when(F.rand(seed + 4) < 0.10, F.lit("bookmark")),
            F.when(F.rand(seed + 5) < 0.08, F.lit("eraser")),
            F.when(F.rand(seed + 6) < 0.06, F.lit("sticky_notes")),
            F.when(F.rand(seed + 7) < 0.05, F.lit("highlighter")),
            F.when(F.rand(seed + 8) < 0.04, F.lit("book")),
        )
    )
    .withColumn("basket_noise", F.expr("filter(basket_noise, x -> x is not null)"))
    .withColumn("items", F.array_distinct(F.array_union(F.col("template_items"), F.col("basket_noise"))))
    .select("transaction_id", "timestamp", "store_id", "customer_type", "basket_type", "items")
)

transactions.cache()
transactions.show(5, truncate=False)
print("Total transactions:", transactions.count())

## 2. FP-Growth Model

We use the Spark MLlib FP-Growth implementation to discover frequent itemsets and association rules.

Suggested thresholds for a large synthetic bookshop dataset:
- `minSupport = 0.01`
- `minConfidence = 0.35`

These settings keep the results meaningful while still capturing common co-purchase patterns.

In [ ]:
fp_growth = FPGrowth(itemsCol="items", minSupport=0.01, minConfidence=0.35)
model = fp_growth.fit(transactions)

freq_itemsets = model.freqItemsets.orderBy(F.desc("freq"))
rules = model.associationRules.orderBy(F.desc("confidence"), F.desc("lift"))

freq_itemsets.show(20, truncate=False)
rules.select("antecedent", "consequent", "confidence", "lift", "support").show(20, truncate=False)

## 3. Business Interpretation

Expected high-value patterns in the simulated bookshop data:

1. **Notebook + Pen**
   - Likely one of the strongest itemsets because stationery bundles are frequent.
2. **Book + Bookmark**
   - Strong reading-related association, useful for cross-selling near fiction and bestseller shelves.
3. **Textbook + Notebook + Highlighter**
   - Useful for student-oriented promotions at the start of the academic term.
4. **Planner + Pen**
   - Good bundle for office and productivity customers.
5. **Diary + Pen** and **Sticky Notes + Pen + Notebook**
   - Strong stationery purchase clusters for impulse placement.

Recommendations:
- Place pens close to notebooks, planners, diaries, and textbook sections.
- Bundle bookmarks with novels and fiction books.
- Promote highlighters and sticky notes alongside textbooks and notebooks.
- Use stock planning to keep stationery items aligned with expected academic-season demand.

In [ ]:
# Optional: save results for reporting
freq_itemsets.write.mode("overwrite").parquet("bookshop_freq_itemsets")
rules.write.mode("overwrite").parquet("bookshop_association_rules")
print("Saved itemsets and rules to parquet outputs.")